In [6]:
#1 Installations

!pip install requests beautifulsoup4 pandas openpyxl

In [3]:
#2 Imports

import requests
import time
import random
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, quote

import xml.etree.ElementTree as ET

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [4]:
#3 Headers

DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [5]:
#4 List of search terms for construction of query-URLs, will later be URL-encoded in the main loop

keywords = [
    "CCS",
    "CCU",
    "CCUS",
    "CCU/S",
    "Carbon Management",
    "Carbon Capture",
    "Carbon Capture Utilisation and Storage",
    "Carbon Capture Utilization and Storage",
    "Carbon Capture and Utilisation",
    "Carbon Capture and Utilization",
    "Abscheid",
    "Abscheidung",
    "Kohlenstoffabscheid",
    "Kohlenstoffabscheidung",
    "Technische Senke",
    "KSpG",
    "KSpTG",
    "Kohlendioxid-Speicherungsgesetz",
    "Kohlendioxid-Speicherung-und-Transport-Gesetz"
]


In [6]:
#5  Categorisation of domain URL types

# A: URL-encoded homepage-searches without pagination
# B: URL-encoded homepage-searches with pagination
# C: sitemap fallback domains (for domains without working URL-encoded homepage search functions)

def get_url_type(url, source_id=None):

    if source_id is not None and source_id in SITEMAP_FALLBACK_DOMAINS:
        return "C"

    if "{PageP}" in url:
        return "B"

    return "A"

In [7]:
#6  List of search URLS

# for domains with tested working URL-encoded homepage-searches, template query URLs were constructed using test searches
# including placeholders {KeywordP} and {PageP} for search terms and results page number

# for type C domains identified at this stage, only the base-urls, like www.domain.de, are used as input
# for type A and B homepages that are auto-promoted to type C, domain will  be extracted in main loop

urls = {
1:"https://www.cemex.de/search?q={KeywordP}&delta=99",
2:"https://bdi.eu/de/suche?page={PageP}&search={KeywordP}", 
3:"https://www.harbourenergy.com/search/?query={KeywordP}&submit=&page={PageP}",
4:"https://green-planet-energy.de/suche?query={KeywordP}&tx_kesearch_pi1%5Bpage%5D={PageP}",
5:"https://www.vci.de",
6:"https://fortschrittinfreiheit.de/page/{PageP}/?s={KeywordP}",
7:"https://suche.agora-industrie.de/suche?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
8:"https://power-shift.de/page/{PageP}/?s={KeywordP}",
9:"https://www.dnr.de/suche?search={KeywordP}&page={PageP}",
10:"https://www.wwf.de/suche?s%5Bpage%5D={PageP}&s%5Bq%5D={KeywordP}",
11:"https://www.nabu.de/modules/suche/htdig.php?htdig%5Bwords%5D={KeywordP}&page={PageP}",
12:"https://www.umweltrat.de/SiteGlobals/Forms/Suche/DE/Servicesuche/Servicesuche_Formular.html?templateQueryString={KeywordP}&resultsPerPage=99",
13:"https://de.bellona.org/page/{PageP}/?s={KeywordP}",
14:"https://www.staedtetag.de/suchergebnisse?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
15:"https://www.ruhr-uni-bochum.de",
16:"https://www.landkreistag.de/suche?q={KeywordP}&start={PageP}0",
17:"https://www.dstgb.de/?q={KeywordP}",
18:"https://www.bund.net/service/suchergebnis/?tx_solr%5Bfilter%5D%5B0%5D=&tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}&tx_solr%5Bsort%5D=relevance+desc",
19:"https://www.lyondellbasell.com/en/search/?keyword={KeywordP}",
20:"https://www.basf.com",
21:"https://www.bde.de/search/?q={KeywordP}",
22:"https://www.eutop.com",
23:"https://www.bund-sachsen.de/service/suchergebnis/?tx_solr%5Bfilter%5D%5B0%5D=&tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}&tx_solr%5Bsort%5D=relevance+desc",
24:"https://www.bund-nrw.de/suchergebnis/?tx_solr%5Bfilter%5D%5B0%5D=&tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}&tx_solr%5Bsort%5D=relevance+desc",
25:"https://www.baustoffindustrie.de/search?query={KeywordP}",
26:"https://www.kalk.de",
27:"https://www.bveg.de/page/{PageP}/?s={KeywordP}&facet_type&sort_type=relevance#038;facet_type&sort_type=relevance",
28:"https://www.campact.de/page/{PageP}/?s={KeywordP}",
29:"https://www.cm-allianz.de",
30:"https://carbon-management-initiative.de",
31:"https://www.duh.de/suche/?no_cache=1&tx_kesearch_pi1%5Bsword%5D={KeywordP}&tx_kesearch_pi1%5Bpage%5D={PageP}",
32:"https://dvne.org",
33:"https://www.verkehrsforum.de/de/suchergebnisse?psearch={KeywordP}",
34:"https://gas-h2.de",
35:"https://www.papierindustrie.de",
36:"https://www.dvgw.de",
37:"https://www.dr-martin-eckert.de",
38:"https://www.thyssenkrupp-uhde.com",
39:"https://www.thyssenkrupp-polysius.com",
40:"https://www.eew-energyfromwaste.com/de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
41:"https://www.equinor.de",
42:"https://corporate.exxonmobil.com/search?search={KeywordP}",
43:"https://www.everllence.com/search-results?search={KeywordP}&search_page={PageP}",
44:"https://www.evonik.com/de/search.html?q={KeywordP}&languages=German%3BEnglish",
45:"https://www.faktor3.de",
46:"https://www.fluxys.com/de/search#search_e={PageP}0&search_q={KeywordP}",
47:"https://forumue.de/page/{PageP}/?s={KeywordP}",
48:"https://www.handundfusser.de",
49:"https://www.germanwatch.org/de/suche?text={KeywordP}",
50:"https://germanzero.de",
51:"https://www.h2ercules.com",
52:"https://www.portofrotterdam.com",
53:"https://www.helmholtz-klima.de/#o=search&q={KeywordP}",
54:"https://www.holcim.de/search-results?search_api_fulltext={KeywordP}&page={PageP}",
55:"https://www.ihk-nord.de",
56:"https://www.itad.de/@@search?sort_on=relevance&sort_order=&SearchableText={KeywordP}&advanced_search=False&pt_toggle=%23&portal_type:list=File&portal_type:list=Newsletter&portal_type:list=Event&portal_type:list=Collection&portal_type:list=Folder&portal_type:list=Image&portal_type:list=Newsletter%20Issue&portal_type:list=Document&portal_type:list=Fragebogen&portal_type:list=Link&portal_type:list=Anlage&_=1773481589563&b_start:int={PageP}0",
57:"https://www.leag.de/de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
58:"https://www.lhoist.com/de-DE/suche?2026prod_search%5Bquery%5D={KeywordP}&2026prod_search%5Bpage%5D={PageP}",
59:"https://www.linde-gas.de",
60:"https://www.mew-verband.de/suche?query={KeywordP}&ccm_paging_p={PageP}&ccm_order_by=&ccm_order_by_direction=",
61:"https://www.gasunie.nl/en/search-the-website?q={KeywordP}&ss360PageLink=1&ss360Offset={PageP}0",
62:"https://www.ontras.com/de/search/node?keys={KeywordP}&page={PageP}",
63:"https://oge.net/de/suche-seite{PageP}?search={KeywordP}",
64:"https://www.rostock-port.de",
65:"https://www.rwe.com/suche/?q={KeywordP}",
66:"https://www.shell.de/search.html?q={KeywordP}&offset={PageP}0",
67:"https://www.swm.de/suche?suchbegriff={KeywordP}",
68:"https://www.thepartners.io",
69:"https://tes-h2.com",
70:"https://www.tuev-nord.de/de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
71:"https://www.tuv.com/germany/de/search.html#?term={KeywordP}&page_offset={PageP}",
72:"https://www.tuev-verband.de/suche?tuev%5Bpage%5D={PageP}&tuev%5Bq%5D={KeywordP}",
73:"https://www.uniper.energy/de/search?query={KeywordP}&page={PageP}",
74:"https://www.urgewald.org/search/content?keys={KeywordP}&page={PageP}",
75:"https://www.vais.de",
76:"https://www.vdma.eu/de/suche#search-page-tabbar-articles-id?searchKeyword={KeywordP}&sorts=RELEVANCE",
77:"https://www.vgms.de/suche?tx_kesearch_pi1%5Bpage%5D={PageP}&tx_kesearch_pi1%5BsortByDir%5D=asc&tx_kesearch_pi1%5Bsword%5D={KeywordP}",
78:"https://www.vdz-online.de",
79:"https://www.vbw-bayern.de/vbw/Suche.jsp.is?queryText={KeywordP}&co=3&pageSize=99&sort=Score",
80:"https://vik.de/search?search={KeywordP}",
81:"https://www.vku.de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
82:"https://www.vbcoll.de",
83:"https://www.efuel-alliance.eu/search?tx_kesearch_pi1%5Bfilter_3_end%5D=&tx_kesearch_pi1%5Bfilter_3_start%5D=&tx_kesearch_pi1%5Bpage%5D={PageP}&tx_kesearch_pi1%5Bsword%5D={KeywordP}",
84:"https://www.bernd-westphal.de/page/{PageP}/?s={KeywordP}",
85:"https://wirtschaftsrat.de",
86:"https://en2x.de/page/{PageP}/?s={KeywordP}",
87:"https://zds-seehaefen.de",
88:"https://www.bayernets.de",
89:"https://www.zdh.de/suchergebnisse/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
}

# type c domains that do not have a working search function. Domains not yielding results in type A and B scraping will be added to fallback in main loop. 
SITEMAP_FALLBACK_DOMAINS = {
    5: "https://www.vci.de",
    15: "https://www.ruhr-uni-bochum.de",
    20: "https://www.basf.com",
    22: "https://www.eutop.com",
    26: "https://www.kalk.de",
    29: "https://www.cm-allianz.de",
    30: "https://carbon-management-initiative.de",
    32: "https://dvne.org",
    34: "https://gas-h2.de",
    35: "https://www.papierindustrie.de",
    36: "https://www.dvgw.de",
    37: "https://www.dr-martin-eckert.de",
    38: "https://www.thyssenkrupp-uhde.com",
    39: "https://www.thyssenkrupp-polysius.com",
    41: "https://www.equinor.de",
    45: "https://www.faktor3.de",
    48: "https://www.handundfusser.de",
    50: "https://germanzero.de",
    51: "https://www.h2ercules.com",
    52: "https://www.portofrotterdam.com",
    55: "https://www.ihk-nord.de",
    59: "https://www.linde-gas.de",
    64: "https://www.rostock-port.de",
    68: "https://www.thepartners.io",
    69: "https://tes-h2.com",
    75: "https://www.vais.de",
    78: "https://www.vdz-online.de",
    82: "https://www.vbcoll.de",
    85: "https://wirtschaftsrat.de",
    87: "https://zds-seehaefen.de",
    88: "https://www.bayernets.de",
    #added after testing result pages html lengths (same lengths for all kinds of keywords: non-working homepage-searches) (See manual_check_cells.ipynb)
    17: "https://www.dstgb.de",
    53: "https://www.helmholtz-klima.de",
    76: "https://www.vdma.eu",
    65: "https://www.rwe.com",
    #added after checking returned links overlap (100% overlap regardless of keyword: non-working homepage-searches) (See manual_check_cells.ipynb)
    44: "https://www.evonik.com",
    21: "https://www.bde.de",
    80: "https://vik.de",
    67: "https://www.swm.de",
    #added after checking returned links overlap (100% overlap or always 0 results: non-working searches) (See manual_check_cells.ipynb)
    43: "https://www.everllence.com",
    46: "https://www.fluxys.com",
    54: "https://www.holcim.de",
    58: "https://www.lhoist.com",
    61: "https://www.gasunie.nl",
    70: "https://www.tuev-nord.de",
    71: "https://www.tuv.com",
    73: "https://www.uniper.energy",
    84: "https://www.bernd-westphal.de",
}


In [8]:
#7 Parser for type A and B URLs

def parse_normal_page(soup, base_url):

    links = []

    main = soup.find("main")

    if not main:
        main = soup

    for tag in main.find_all(href=True):

        href = tag["href"]

        if href.startswith(("#","javascript:","mailto:","tel:")):  # Safe Filter: Skip within-page anchors (#), java script executions, email adresses, phone numbers 
            continue

        absolute = urljoin(base_url, href)

        links.append(absolute)

    return links

In [9]:
#8 Parser / helper for sitemap fallback (Type C)
# and search term variants for sitemap URL-matching
# and safe filter helper for Type A and B

from urllib.parse import urlparse, urljoin

KEYWORD_URL_VARIANTS_CACHE = {} # Cache for broader URL matching including defined search term variants


def parse_sitemap_xml(xml_text, sitemap_url=None):
    root = ET.fromstring(xml_text)

    urls = []
    sitemaps = []

    tag = root.tag.lower()

    if "sitemapindex" in tag:
        for elem in root.iter():
            if elem.tag.lower().endswith("loc") and elem.text:
                loc = elem.text.strip()

                loc = loc.replace("https%3A//", "https://").replace("http%3A//", "http://")

                second_https = loc.find("https://", 8)
                second_http = loc.find("http://", 7)
                candidates = [pos for pos in [second_https, second_http] if pos != -1]
                if candidates:
                    loc = loc[min(candidates):]

                if sitemap_url is not None and loc.startswith("/"):
                    loc = urljoin(sitemap_url, loc)

                sitemaps.append(loc)

    elif "urlset" in tag:
        for elem in root.iter():
            if elem.tag.lower().endswith("loc") and elem.text:
                loc = elem.text.strip()

                loc = loc.replace("https%3A//", "https://").replace("http%3A//", "http://")

                second_https = loc.find("https://", 8)
                second_http = loc.find("http://", 7)
                candidates = [pos for pos in [second_https, second_http] if pos != -1]
                if candidates:
                    loc = loc[min(candidates):]

                if sitemap_url is not None and loc.startswith("/"):
                    loc = urljoin(sitemap_url, loc)

                urls.append(loc)

    return sitemaps, urls


def fetch_with_ssl_retry(url, timeout=10):      # for SSL retry handling if pages cannot be accessed due to SSL verification issues (all websites tested to be trustworthy)
    ssl_retry_used = False

    try:
        resp = requests.get(
            url,
            timeout=timeout,
            verify=True,
            headers=DEFAULT_HEADERS
        )
        resp.raise_for_status()
        return resp, ssl_retry_used

    except requests.exceptions.SSLError:
        resp = requests.get(
            url,
            timeout=timeout,
            verify=False,
            headers=DEFAULT_HEADERS
        )
        resp.raise_for_status()

        ssl_retry_domains.add(urlparse(url).netloc)
        ssl_retry_urls.append(url)
        ssl_retry_used = True

        print(f"   ssl retry used: {url}")
        return resp, ssl_retry_used


def get_sitemap_candidates(base_url):
    base_url = base_url.rstrip("/")

    return [
        f"{base_url}/sitemap.xml",
        f"{base_url}/sitemap_index.xml",
        f"{base_url}/sitemap-index.xml"
    ]


def get_robots_sitemaps(base_url, timeout=10):
    robots_url = f"{base_url.rstrip('/')}/robots.txt"
    found = []
    ssl_retry_used = False
    error = None

    try:
        resp, ssl_retry_used = fetch_with_ssl_retry(robots_url, timeout=timeout)

        for line in resp.text.splitlines():
            line = line.strip()
            if line.lower().startswith("sitemap:"):
                sitemap_url = line.split(":", 1)[1].strip()
                if sitemap_url:
                    found.append(sitemap_url)

    except Exception as e:
        error = f"{type(e).__name__}: {e}"
        print(f"   robots.txt read failed: {robots_url} | {repr(e)}")

    return {
        "robots_url": robots_url,
        "sitemaps": found,
        "ssl_retry_used": ssl_retry_used,
        "error": error
    }


def collect_urls_from_sitemap(start_url, max_urls=10000, timeout=10, visited=None):
    if visited is None:
        visited = set()

    to_visit = [start_url]
    collected_urls = []
    ssl_retry_used_any = False
    attempted_sitemaps = []
    parse_errors = []

    while to_visit and len(collected_urls) < max_urls:
        sitemap_url = to_visit.pop(0)

        if sitemap_url in visited:
            continue
        visited.add(sitemap_url)
        attempted_sitemaps.append(sitemap_url)

        try:
            resp, ssl_retry_used = fetch_with_ssl_retry(sitemap_url, timeout=timeout)
            ssl_retry_used_any = ssl_retry_used_any or ssl_retry_used

        except Exception as e:
            print(f"   sitemap read failed: {sitemap_url} | {repr(e)}")
            parse_errors.append({
                "sitemap_url": sitemap_url,
                "stage": "request",
                "error": f"{type(e).__name__}: {e}"
            })
            continue

        try:
            new_sitemaps, page_urls = parse_sitemap_xml(resp.text, sitemap_url=sitemap_url)
        except Exception as e:
            print(f"   xml parse failed: {sitemap_url} | {repr(e)}")
            parse_errors.append({
                "sitemap_url": sitemap_url,
                "stage": "xml_parse",
                "error": f"{type(e).__name__}: {e}"
            })
            continue

        for new_sm in new_sitemaps:
            if new_sm not in visited and new_sm not in to_visit:
                to_visit.append(new_sm)

        for u in page_urls:
            collected_urls.append(u)
            if len(collected_urls) >= max_urls:
                break

    return {
        "page_urls": collected_urls,
        "ssl_retry_used": ssl_retry_used_any,
        "attempted_sitemaps": attempted_sitemaps,
        "parse_errors": parse_errors
    }


def collect_urls_for_domain(base_domain, source_id=None, max_urls=10000, timeout=10):
    manual_candidates = []
    if source_id is not None and source_id in MANUAL_TYPE_C_SITEMAPS:
        manual_candidates = MANUAL_TYPE_C_SITEMAPS[source_id]

    robots_result = get_robots_sitemaps(base_domain, timeout=timeout)
    direct_candidates = get_sitemap_candidates(base_domain)

    ordered_candidates = []

    for c in manual_candidates:
        ordered_candidates.append((c, "manual"))

    for c in robots_result["sitemaps"]:
        if c not in [x[0] for x in ordered_candidates]:
            ordered_candidates.append((c, "robots_txt"))

    for c in direct_candidates:
        if c not in [x[0] for x in ordered_candidates]:
            ordered_candidates.append((c, "direct_candidate"))

    print("   Candidate sitemaps:")
    for c, source in ordered_candidates:
        print(f"   - {c} [{source}]")

    visited = set()
    all_page_urls = []
    sitemap_url_used = None
    sitemap_source = None
    last_error = None
    ssl_retry_used_any = robots_result["ssl_retry_used"]

    for candidate_url, candidate_source in ordered_candidates:
        result = collect_urls_from_sitemap(
            candidate_url,
            max_urls=max_urls,
            timeout=timeout,
            visited=visited
        )

        ssl_retry_used_any = ssl_retry_used_any or result["ssl_retry_used"]

        if result["parse_errors"]:
            last_error = result["parse_errors"][-1]["error"]

        if result["page_urls"]:
            all_page_urls = result["page_urls"]
            sitemap_url_used = candidate_url
            sitemap_source = candidate_source
            break

    debug_info = {
        "source_id": source_id,
        "base_domain": base_domain,
        "sitemap_source": sitemap_source,
        "sitemap_url_used": sitemap_url_used,
        "n_sitemap_urls_collected": len(all_page_urls),
        "n_candidate_sitemaps": len(ordered_candidates),
        "robots_txt_error": robots_result["error"],
        "ssl_retry_used": ssl_retry_used_any,
        "discovery_status": "ok" if len(all_page_urls) > 0 else "no_working_sitemap",
        "discovery_error": last_error
    }

    return all_page_urls, debug_info


# --------------------------------------------------
# A/B SAFE FILTER HELPERS (iteratively set reviewing raw search results output)
# --------------------------------------------------

SAFE_EXCLUDE_SUBSTRINGS = [
    "/wp-content/",
    "/wp-includes/",
    "/assets/",
    "/static/",
    "/plugins/",
    "/themes/",
    "/css/",
    "/js/",
    "/images/",
    "/img/",
    "/fonts/",
    "/fileadmin/",
    "/siteglobals/",
    "/_resources/",
    "/servicesuche",
    "/forms",
    "/suche",
    "/search",
    "/@@search",
    "/newsletter",
    "/spenden",
    "/mitglied",
    "/mitglied-werden",
    "/login",
    "/signin",
    "/signup",
    "/register",
    "/account",
    "/kontakt",
    "/contact",
    "/impressum",
    "/datenschutz",
    "/privacy",
    "/karriere",
    "/jobs",
    "/job",
    "/tag/",
    "/category/",
    "/author/",
    "/profile/",
    "/profiles/",
]

SAFE_EXCLUDE_FILE_ENDINGS = [
    ".css",
    ".js",
    ".png",
    ".jpg",
    ".jpeg",
    ".svg",
    ".gif",
    ".ico",
    ".webp",
]

SAFE_EXCLUDE_DOMAINS = [
    "youtube.com",
    "www.youtube.com",
    "facebook.com",
    "www.facebook.com",
    "linkedin.com",
    "www.linkedin.com",
    "instagram.com",
    "www.instagram.com",
    "x.com",
    "www.x.com",
    "twitter.com",
    "www.twitter.com",
    "tiktok.com",
    "www.tiktok.com",
]

SAFE_EXCLUDE_QUERY_PARTS = [
    "utm_",
    "fbclid=",
    "gclid=",
    "share=",
    "print=",
    "reply=",
    "comment",
]


def get_safe_filter_reason(url):
    if pd.isna(url):
        return "missing_url"

    url_l = str(url).lower().strip()
    parsed = urlparse(url_l)

    
    if parsed.path.endswith(".pdf"):   # explicitly keep urls ending with .pdf in search for publication uploads
        return None

    for ending in SAFE_EXCLUDE_FILE_ENDINGS:
        if parsed.path.endswith(ending):
            return f"safe_file_ending:{ending}"

    domain = parsed.netloc.lower()
    for bad_domain in SAFE_EXCLUDE_DOMAINS:
        if domain == bad_domain or domain.endswith("." + bad_domain):
            return f"safe_external_domain:{bad_domain}"

    if parsed.query:
        for q in SAFE_EXCLUDE_QUERY_PARTS:
            if q in parsed.query:
                return f"safe_query:{q}"

    path_check = parsed.path.lower()
    for pattern in SAFE_EXCLUDE_SUBSTRINGS:
        if pattern in path_check:
            return f"safe_path:{pattern}"

    return None


def get_keyword_url_variants(keyword):
    keyword_l = keyword.lower().strip()

    if keyword_l in KEYWORD_URL_VARIANTS_CACHE:
        return KEYWORD_URL_VARIANTS_CACHE[keyword_l]

    manual_variants = {
        "ccs": [
            "ccs",
            "carbon-capture-storage",
            "carbon-capture-and-storage",
            "carboncapturestorage",
            "co2-storage",
            "co2-speicherung",
        ],
        "ccu": [
            "ccu",
            "carbon-capture-utilization",
            "carbon-capture-and-utilization",
            "carboncaptureutilization",
            "co2-utilization",
            "co2-nutzung",
        ],
        "ccus": [
            "ccus",
            "carbon-capture-utilization-storage",
            "carbon-capture-utilisation-storage",
            "carbon-capture-and-storage",
            "carbon-capture-storage",
        ],
        "carbon capture": [
            "carbon-capture",
            "carboncapture",
            "capture",
            "capturing",
            "captured",
            "co2-capture",
        ],
        "carbon management": [
            "carbon-management",
            "carbonmanagement",
            "co2-management",
            "carbon-manager",
        ],
        "abscheidung": [
            "abscheidung",
            "co2-abscheidung",
            "capture",
            "capturing",
            "captured",
        ],
        "kohlendioxid-speicherung": [
            "kohlendioxid-speicherung",
            "co2-speicherung",
            "carbon-storage",
            "storage",
        ],
        "kohlendioxid-speicherung-und-transport-gesetz": [
            "kohlendioxid-speicherung-und-transport-gesetz",
            "ksptg",
            "kspg",
        ],
        "kspg": [
            "kspg",
            "kohlendioxid-speicherungsgesetz",
        ],
        "ksptg": [
            "ksptg",
            "kohlendioxid-speicherung-und-transport-gesetz",
        ],
    }

    variants = set()
    variants.add(keyword_l)
    variants.add(keyword_l.replace(" ", "-"))
    variants.add(keyword_l.replace(" ", ""))
    variants.add(keyword_l.replace(" ", "_"))

    if keyword_l in manual_variants:
        variants.update(manual_variants[keyword_l])

    variants = sorted({v.strip() for v in variants if v and str(v).strip()})
    KEYWORD_URL_VARIANTS_CACHE[keyword_l] = variants
    return variants


def url_keyword_match(url, keyword):
    url_l = url.lower()
    variants = get_keyword_url_variants(keyword)

    for variant in variants:
        patterns = [
            f"/{variant}",
            f"-{variant}",
            f"_{variant}",
            f"{variant}/",
            f"{variant}-",
            f"{variant}_",
        ]

        if any(p in url_l for p in patterns):
            return True

        if variant in url_l:
            return True

    return False


def find_keyword_hits_in_urls(urls, keyword, max_hits=20):
    hits = []
    seen = set()

    for url in urls:
        if url_keyword_match(url, keyword):
            if url not in seen:
                hits.append(url)
                seen.add(url)

        if len(hits) >= max_hits:
            break

    return hits

In [10]:
#9 SETUP for main loop results and logging
# and link occurrence tracking for ubiquitous-link filtering

from collections import defaultdict, Counter

# Results container
scraped_data = []
failed_urls = []
zero_link_urls = []

# Separate exclusion log for filtered links (A/B)
filtered_out_links = []

# SSL-Retry Setup and Cache
ssl_retry_domains = set()
ssl_retry_urls = []

# Cache for sitemap page URLs per source_id
sitemap_url_cache = {}

# Debug metadata for sitemap discovery per source_id
sitemap_debug_cache = {}

# Tracking for automatic promotion of domains to sitemap fallback
type_a_run_stats = {}
type_b_run_stats = {}
auto_added_to_fallback_domains = {}

# Detailed fallback log for documentation
type_c_fallback_log = []

# Separate discovery failure log for Type C
type_c_discovery_failures = []

# --------------------------------------------------
# A/B link occurrence tracking for ubiquitous-link filtering
# --------------------------------------------------
# Counts how often a found_url appears across all A/B requests of a domain
ab_link_occurrence_counter = defaultdict(Counter)

# Counts how many A/B requests were actually processed per domain
ab_request_counter = defaultdict(int)


# Helper for SSL-exceptions / SSL retry handling if pages cannot be accessed due to SSL verification issues (all websites tested to be trustworthy)
def get_request_kwargs(url):
    domain = urlparse(url).netloc

    if domain in ssl_retry_domains:
        return {
            "verify": False,
            "headers": DEFAULT_HEADERS
        }

    return {
        "verify": True,
        "headers": DEFAULT_HEADERS
    }

In [13]:
#10  ------------------- MAIN LOOP SCRAPER -------------------------------

counter = 0
start_time = time.time()

# Setting sleep time between requests (to decrease blocking)
SLEEP_TYPE_A = 0.3 
SLEEP_TYPE_B = 0.3  
SLEEP_TYPE_C = 0.1 

# Setting search result pages to be scraped for type b (set to 1 for debugging) (final run: 0, 1, 2, 3)
type_b_pages = [0, 1, 2, 3]


# Debugging: Toggle switches for URL types (set to "True" to run loop for URL type, set to "False" to skip URL type in main loop)
RUN_TYPE_A = False
RUN_TYPE_B = False
RUN_TYPE_C = False

for source_id, url_template in urls.items():

    url_type = get_url_type(url_template, source_id)

    if url_type == "A" and not RUN_TYPE_A:
        continue
    if url_type == "B" and not RUN_TYPE_B:
        continue
    if url_type == "C" and not RUN_TYPE_C:
        continue

    # ------------------------------------------------------------------
    # TYPE A SETUP
    # ------------------------------------------------------------------
    if url_type == "A" and source_id not in type_a_run_stats:
        type_a_run_stats[source_id] = {
            "url": url_template,
            "tested_keywords": [],
            "total_hits": 0,
            "fallback_checked": False,
            "expected_requests": len(keywords),
        }

    # ------------------------------------------------------------------
    # TYPE B SETUP
    # ------------------------------------------------------------------
    if url_type == "B" and source_id not in type_b_run_stats:
        type_b_run_stats[source_id] = {
            "url": url_template,
            "tested_combinations": [],
            "total_hits": 0,
            "fallback_checked": False,
            "expected_combinations": len(keywords) * len(type_b_pages)
        }

    for keyword in keywords:

        keyword_enc = quote(keyword, safe="")

        # ==============================================================
        # TYPE A
        # ==============================================================
        if url_type == "A" and RUN_TYPE_A:

            page_value = None
            full_url = url_template.format(KeywordP=keyword_enc)
            print(f"[{source_id}] A | KW='{keyword}'")

            response = None
            ssl_retry_used = False

            try:
                kwargs = get_request_kwargs(full_url)

                try:
                    response = requests.get(full_url, timeout=10, **kwargs)

                except requests.exceptions.SSLError as e:
                    print(f"⚠️ SSL-Fehler -> Retry ohne Verifikation: {full_url}")
                    print(f"   → {type(e).__name__}: {e}")

                    response = requests.get(
                        full_url,
                        timeout=10,
                        verify=False,
                        headers=DEFAULT_HEADERS
                    )

                    ssl_retry_domains.add(urlparse(full_url).netloc)
                    ssl_retry_urls.append(full_url)
                    ssl_retry_used = True

                soup = BeautifulSoup(response.text, "html.parser")
                links_raw = parse_normal_page(soup, full_url)

                # ---------- SAFE FILTER A ----------
                links_kept = []
                seen_kept_in_request = set()

                for link in links_raw:
                    filter_reason = get_safe_filter_reason(link)

                    if filter_reason is not None:
                        filtered_out_links.append({
                            "source_id": source_id,
                            "type": "A",
                            "keyword": keyword,
                            "page": page_value,
                            "search_url": full_url,
                            "found_url": link,
                            "filter_reason": filter_reason,
                            "filter_stage": "safe_filter_ab",
                            "ssl_retry_used": ssl_retry_used
                        })
                    else:
                        if link not in seen_kept_in_request:
                            links_kept.append(link)
                            seen_kept_in_request.add(link)

                # occurrence tracking only for kept links, once per request
                for link in seen_kept_in_request:
                    ab_link_occurrence_counter[source_id][link] += 1

                ab_request_counter[source_id] += 1

                type_a_run_stats[source_id]["tested_keywords"].append(keyword)
                type_a_run_stats[source_id]["total_hits"] += len(links_kept)

                print(
                    f"   → raw={len(links_raw)} | kept={len(links_kept)} | "
                    f"safe_filtered={len(links_raw) - len(links_kept)} | "
                    f"ssl_retry_used={ssl_retry_used}"
                )

                if len(links_kept) == 0:
                    zero_link_urls.append({
                        "source_id": source_id,
                        "type": "A",
                        "keyword": keyword,
                        "page": page_value,
                        "search_url": full_url,
                        "ssl_retry_used": ssl_retry_used
                    })

                for link in links_kept:
                    scraped_data.append({
                        "source_id": source_id,
                        "keyword": keyword,
                        "page": page_value,
                        "search_url": full_url,
                        "found_url": link,
                        "ssl_retry_used": ssl_retry_used
                    })

            except Exception as e:
                print(f"❌ Fehler bei {full_url}")
                print(f"   → {type(e).__name__}: {e}")

                type_a_run_stats[source_id]["tested_keywords"].append(keyword)
                ab_request_counter[source_id] += 1

                failed_urls.append({
                    "url": full_url,
                    "error": str(e),
                    "type": "A",
                    "keyword": keyword,
                    "page": page_value,
                    "status_code": response.status_code if response is not None else None,
                    "ssl_retry_used": ssl_retry_used
                })

            # --------- POST-PROCESS A UBIQUITOUS LINKS ---------
            if len(type_a_run_stats[source_id]["tested_keywords"]) == type_a_run_stats[source_id]["expected_requests"]:
                expected_count = ab_request_counter[source_id]
                ubiquitous_links = {
                    link for link, cnt in ab_link_occurrence_counter[source_id].items()
                    if cnt == expected_count and expected_count > 0
                }

                if len(ubiquitous_links) > 0:
                    kept_scraped_data = []

                    for row in scraped_data:
                        if (
                            row["source_id"] == source_id
                            and row["search_url"].startswith(url_template.split("{", 1)[0])
                            and row["found_url"] in ubiquitous_links
                        ):
                            filtered_out_links.append({
                                "source_id": row["source_id"],
                                "type": "A",
                                "keyword": row["keyword"],
                                "page": row["page"],
                                "search_url": row["search_url"],
                                "found_url": row["found_url"],
                                "filter_reason": "ubiquitous_ab_link",
                                "filter_stage": "post_domain_ab_filter",
                                "ssl_retry_used": row["ssl_retry_used"]
                            })
                        else:
                            kept_scraped_data.append(row)

                    scraped_data = kept_scraped_data
                    type_a_run_stats[source_id]["total_hits"] -= len(ubiquitous_links) * len(keywords)
                    if type_a_run_stats[source_id]["total_hits"] < 0:
                        type_a_run_stats[source_id]["total_hits"] = 0

                    print(f"   → A ubiquitous links removed for source_id={source_id}: {len(ubiquitous_links)}")

            # --------- AUTO-PROMOTION A -> C ---------
            if (
                len(type_a_run_stats[source_id]["tested_keywords"]) == len(keywords)
                and not type_a_run_stats[source_id]["fallback_checked"]
            ):

                if type_a_run_stats[source_id]["total_hits"] == 0:
                    SITEMAP_FALLBACK_DOMAINS[source_id] = url_template
                    auto_added_to_fallback_domains[source_id] = url_template
                    type_a_run_stats[source_id]["fallback_checked"] = True

                    base_domain = url_template.split("?", 1)[0]
                    base_domain = base_domain.split("#", 1)[0]

                    parsed = urlparse(base_domain)
                    base_domain = f"{parsed.scheme}://{parsed.netloc}"

                    print(f"   → Auto-promoted to sitemap fallback: {source_id}")
                    print(f"   → Fallback base domain: {base_domain}")

                    ssl_retry_count_before = len(ssl_retry_urls)
                    sitemap_debug = {}

                    try:
                        if source_id in sitemap_url_cache:
                            all_page_urls = sitemap_url_cache[source_id]
                            sitemap_debug = sitemap_debug_cache[source_id]
                            print(f"   → Using cached sitemap URLs: {len(all_page_urls)}")
                        else:
                            all_page_urls, sitemap_debug = collect_urls_for_domain(
                                base_domain,
                                source_id=source_id,
                                max_urls=10000,
                                timeout=10
                            )
                            sitemap_url_cache[source_id] = all_page_urls
                            sitemap_debug_cache[source_id] = sitemap_debug

                            print(f"   → Collected page URLs: {len(all_page_urls)}")
                            print(f"   → Sitemap source: {sitemap_debug['sitemap_source']}")
                            print(f"   → Sitemap used: {sitemap_debug['sitemap_url_used']}")
                            print(f"   → Discovery status: {sitemap_debug['discovery_status']}")

                        fallback_ssl_retry_used = (
                            sitemap_debug["ssl_retry_used"] or
                            (len(ssl_retry_urls) > ssl_retry_count_before)
                        )

                        for fallback_keyword in keywords:
                            fallback_links = find_keyword_hits_in_urls(
                                all_page_urls,
                                fallback_keyword,
                                max_hits=20
                            )

                            print(
                                f"   → Fallback KW='{fallback_keyword}': "
                                f"{len(fallback_links)} URL Hits | "
                                f"ssl_retry_used={fallback_ssl_retry_used}"
                            )

                            type_c_fallback_log.append({
                                "source_id": source_id,
                                "origin_type": "A_auto_promoted",
                                "base_domain": base_domain,
                                "keyword": fallback_keyword,
                                "status": (
                                    "no_working_sitemap"
                                    if sitemap_debug["discovery_status"] == "no_working_sitemap"
                                    else ("hits_found" if len(fallback_links) > 0 else "no_hits")
                                ),
                                "n_hits": len(fallback_links),
                                "ssl_retry_used": fallback_ssl_retry_used,
                                "sitemap_urls_collected": len(all_page_urls),
                                "sitemap_source": sitemap_debug["sitemap_source"],
                                "sitemap_url_used": sitemap_debug["sitemap_url_used"],
                                "n_candidate_sitemaps": sitemap_debug["n_candidate_sitemaps"],
                                "robots_txt_error": sitemap_debug["robots_txt_error"],
                                "discovery_status": sitemap_debug["discovery_status"],
                                "discovery_error": sitemap_debug["discovery_error"],
                                "error": None
                            })

                            if len(fallback_links) == 0:
                                zero_link_urls.append({
                                    "source_id": source_id,
                                    "type": "C",
                                    "keyword": fallback_keyword,
                                    "page": None,
                                    "search_url": base_domain,
                                    "ssl_retry_used": fallback_ssl_retry_used
                                })

                            for link in fallback_links:
                                scraped_data.append({
                                    "source_id": source_id,
                                    "keyword": fallback_keyword,
                                    "page": None,
                                    "search_url": base_domain,
                                    "found_url": link,
                                    "ssl_retry_used": fallback_ssl_retry_used
                                })

                    except Exception as e:
                        print(f"❌ Fehler beim direkten Sitemap-Fallback: {base_domain}")
                        print(f"   → {type(e).__name__}: {e}")

                        type_c_fallback_log.append({
                            "source_id": source_id,
                            "origin_type": "A_auto_promoted",
                            "base_domain": base_domain,
                            "keyword": None,
                            "status": "fallback_failed",
                            "n_hits": None,
                            "ssl_retry_used": len(ssl_retry_urls) > ssl_retry_count_before,
                            "sitemap_urls_collected": None,
                            "sitemap_source": sitemap_debug.get("sitemap_source") if sitemap_debug else None,
                            "sitemap_url_used": sitemap_debug.get("sitemap_url_used") if sitemap_debug else None,
                            "n_candidate_sitemaps": sitemap_debug.get("n_candidate_sitemaps") if sitemap_debug else None,
                            "robots_txt_error": sitemap_debug.get("robots_txt_error") if sitemap_debug else None,
                            "discovery_status": sitemap_debug.get("discovery_status") if sitemap_debug else None,
                            "discovery_error": sitemap_debug.get("discovery_error") if sitemap_debug else None,
                            "error": f"{type(e).__name__}: {e}"
                        })

                        failed_urls.append({
                            "url": base_domain,
                            "error": str(e),
                            "type": "C",
                            "keyword": None,
                            "page": None,
                            "status_code": None,
                            "ssl_retry_used": len(ssl_retry_urls) > ssl_retry_count_before
                        })

                else:
                    type_a_run_stats[source_id]["fallback_checked"] = True

            time.sleep(SLEEP_TYPE_A)
            counter += 1

        # ==============================================================
        # TYPE B
        # ==============================================================
        elif url_type == "B" and RUN_TYPE_B:

            for page in type_b_pages:

                page_value = page
                full_url = url_template.format(
                    KeywordP=keyword_enc,
                    PageP=page_value
                )

                print(f"[{source_id}] B | KW='{keyword}' | Page={page_value}")

                response = None
                ssl_retry_used = False

                try:
                    kwargs = get_request_kwargs(full_url)

                    try:
                        response = requests.get(full_url, timeout=10, **kwargs)

                    except requests.exceptions.SSLError as e:
                        print(f"⚠️ SSL-Fehler -> Retry ohne Verifikation: {full_url}")
                        print(f"   → {type(e).__name__}: {e}")

                        response = requests.get(
                            full_url,
                            timeout=10,
                            verify=False,
                            headers=DEFAULT_HEADERS
                        )

                        ssl_retry_domains.add(urlparse(full_url).netloc)
                        ssl_retry_urls.append(full_url)
                        ssl_retry_used = True

                    soup = BeautifulSoup(response.text, "html.parser")
                    links_raw = parse_normal_page(soup, full_url)

                    # ---------- SAFE FILTER B ----------
                    links_kept = []
                    seen_kept_in_request = set()

                    for link in links_raw:
                        filter_reason = get_safe_filter_reason(link)

                        if filter_reason is not None:
                            filtered_out_links.append({
                                "source_id": source_id,
                                "type": "B",
                                "keyword": keyword,
                                "page": page_value,
                                "search_url": full_url,
                                "found_url": link,
                                "filter_reason": filter_reason,
                                "filter_stage": "safe_filter_ab",
                                "ssl_retry_used": ssl_retry_used
                            })
                        else:
                            if link not in seen_kept_in_request:
                                links_kept.append(link)
                                seen_kept_in_request.add(link)

                    for link in seen_kept_in_request:
                        ab_link_occurrence_counter[source_id][link] += 1

                    ab_request_counter[source_id] += 1

                    type_b_run_stats[source_id]["tested_combinations"].append((keyword, page_value))
                    type_b_run_stats[source_id]["total_hits"] += len(links_kept)

                    print(
                        f"   → raw={len(links_raw)} | kept={len(links_kept)} | "
                        f"safe_filtered={len(links_raw) - len(links_kept)} | "
                        f"ssl_retry_used={ssl_retry_used}"
                    )

                    if len(links_kept) == 0:
                        zero_link_urls.append({
                            "source_id": source_id,
                            "type": "B",
                            "keyword": keyword,
                            "page": page_value,
                            "search_url": full_url,
                            "ssl_retry_used": ssl_retry_used
                        })

                    for link in links_kept:
                        scraped_data.append({
                            "source_id": source_id,
                            "keyword": keyword,
                            "page": page_value,
                            "search_url": full_url,
                            "found_url": link,
                            "ssl_retry_used": ssl_retry_used
                        })

                except Exception as e:
                    print(f"❌ Fehler bei: {full_url}")
                    print(f"   → {type(e).__name__}: {e}")

                    type_b_run_stats[source_id]["tested_combinations"].append((keyword, page_value))
                    ab_request_counter[source_id] += 1

                    failed_urls.append({
                        "url": full_url,
                        "error": str(e),
                        "type": "B",
                        "keyword": keyword,
                        "page": page_value,
                        "status_code": response.status_code if response is not None else None,
                        "ssl_retry_used": ssl_retry_used
                    })

                # --------- POST-PROCESS B UBIQUITOUS LINKS ---------
                if len(type_b_run_stats[source_id]["tested_combinations"]) == type_b_run_stats[source_id]["expected_combinations"]:
                    expected_count = ab_request_counter[source_id]
                    ubiquitous_links = {
                        link for link, cnt in ab_link_occurrence_counter[source_id].items()
                        if cnt == expected_count and expected_count > 0
                    }

                    if len(ubiquitous_links) > 0:
                        kept_scraped_data = []

                        for row in scraped_data:
                            if (
                                row["source_id"] == source_id
                                and row["found_url"] in ubiquitous_links
                            ):
                                filtered_out_links.append({
                                    "source_id": row["source_id"],
                                    "type": "B",
                                    "keyword": row["keyword"],
                                    "page": row["page"],
                                    "search_url": row["search_url"],
                                    "found_url": row["found_url"],
                                    "filter_reason": "ubiquitous_ab_link",
                                    "filter_stage": "post_domain_ab_filter",
                                    "ssl_retry_used": row["ssl_retry_used"]
                                })
                            else:
                                kept_scraped_data.append(row)

                        scraped_data = kept_scraped_data

                        remaining_b_hits = sum(
                            1 for row in scraped_data
                            if row["source_id"] == source_id
                        )
                        type_b_run_stats[source_id]["total_hits"] = remaining_b_hits

                        print(f"   → B ubiquitous links removed for source_id={source_id}: {len(ubiquitous_links)}")

                # --------- AUTO-PROMOTION B -> C ---------
                if (
                    len(type_b_run_stats[source_id]["tested_combinations"]) == type_b_run_stats[source_id]["expected_combinations"]
                    and not type_b_run_stats[source_id]["fallback_checked"]
                ):

                    if type_b_run_stats[source_id]["total_hits"] == 0:
                        SITEMAP_FALLBACK_DOMAINS[source_id] = url_template
                        auto_added_to_fallback_domains[source_id] = url_template
                        type_b_run_stats[source_id]["fallback_checked"] = True

                        base_domain = url_template.split("?", 1)[0]
                        base_domain = base_domain.split("#", 1)[0]

                        parsed = urlparse(base_domain)
                        base_domain = f"{parsed.scheme}://{parsed.netloc}"

                        print(f"   → Auto-promoted to sitemap fallback: {source_id}")
                        print(f"   → Fallback base domain: {base_domain}")

                        ssl_retry_count_before = len(ssl_retry_urls)
                        sitemap_debug = {}

                        try:
                            if source_id in sitemap_url_cache:
                                all_page_urls = sitemap_url_cache[source_id]
                                sitemap_debug = sitemap_debug_cache[source_id]
                                print(f"   → Using cached sitemap URLs: {len(all_page_urls)}")
                            else:
                                all_page_urls, sitemap_debug = collect_urls_for_domain(
                                    base_domain,
                                    source_id=source_id,
                                    max_urls=10000,
                                    timeout=10
                                )
                                sitemap_url_cache[source_id] = all_page_urls
                                sitemap_debug_cache[source_id] = sitemap_debug

                                print(f"   → Collected page URLs: {len(all_page_urls)}")
                                print(f"   → Sitemap source: {sitemap_debug['sitemap_source']}")
                                print(f"   → Sitemap used: {sitemap_debug['sitemap_url_used']}")
                                print(f"   → Discovery status: {sitemap_debug['discovery_status']}")

                            fallback_ssl_retry_used = (
                                sitemap_debug["ssl_retry_used"] or
                                (len(ssl_retry_urls) > ssl_retry_count_before)
                            )

                            for fallback_keyword in keywords:
                                fallback_links = find_keyword_hits_in_urls(
                                    all_page_urls,
                                    fallback_keyword,
                                    max_hits=20
                                )

                                print(
                                    f"   → Fallback KW='{fallback_keyword}': "
                                    f"{len(fallback_links)} URL Hits | "
                                    f"ssl_retry_used={fallback_ssl_retry_used}"
                                )

                                type_c_fallback_log.append({
                                    "source_id": source_id,
                                    "origin_type": "B_auto_promoted",
                                    "base_domain": base_domain,
                                    "keyword": fallback_keyword,
                                    "status": (
                                        "no_working_sitemap"
                                        if sitemap_debug["discovery_status"] == "no_working_sitemap"
                                        else ("hits_found" if len(fallback_links) > 0 else "no_hits")
                                    ),
                                    "n_hits": len(fallback_links),
                                    "ssl_retry_used": fallback_ssl_retry_used,
                                    "sitemap_urls_collected": len(all_page_urls),
                                    "sitemap_source": sitemap_debug["sitemap_source"],
                                    "sitemap_url_used": sitemap_debug["sitemap_url_used"],
                                    "n_candidate_sitemaps": sitemap_debug["n_candidate_sitemaps"],
                                    "robots_txt_error": sitemap_debug["robots_txt_error"],
                                    "discovery_status": sitemap_debug["discovery_status"],
                                    "discovery_error": sitemap_debug["discovery_error"],
                                    "error": None
                                })

                                if len(fallback_links) == 0:
                                    zero_link_urls.append({
                                        "source_id": source_id,
                                        "type": "C",
                                        "keyword": fallback_keyword,
                                        "page": None,
                                        "search_url": base_domain,
                                        "ssl_retry_used": fallback_ssl_retry_used
                                    })

                                for link in fallback_links:
                                    scraped_data.append({
                                        "source_id": source_id,
                                        "keyword": fallback_keyword,
                                        "page": None,
                                        "search_url": base_domain,
                                        "found_url": link,
                                        "ssl_retry_used": fallback_ssl_retry_used
                                    })

                        except Exception as e:
                            print(f"❌ Fehler beim direkten Sitemap-Fallback: {base_domain}")
                            print(f"   → {type(e).__name__}: {e}")

                            type_c_fallback_log.append({
                                "source_id": source_id,
                                "origin_type": "B_auto_promoted",
                                "base_domain": base_domain,
                                "keyword": None,
                                "status": "fallback_failed",
                                "n_hits": None,
                                "ssl_retry_used": len(ssl_retry_urls) > ssl_retry_count_before,
                                "sitemap_urls_collected": None,
                                "sitemap_source": sitemap_debug.get("sitemap_source") if sitemap_debug else None,
                                "sitemap_url_used": sitemap_debug.get("sitemap_url_used") if sitemap_debug else None,
                                "n_candidate_sitemaps": sitemap_debug.get("n_candidate_sitemaps") if sitemap_debug else None,
                                "robots_txt_error": sitemap_debug.get("robots_txt_error") if sitemap_debug else None,
                                "discovery_status": sitemap_debug.get("discovery_status") if sitemap_debug else None,
                                "discovery_error": sitemap_debug.get("discovery_error") if sitemap_debug else None,
                                "error": f"{type(e).__name__}: {e}"
                            })

                            failed_urls.append({
                                "url": base_domain,
                                "error": str(e),
                                "type": "C",
                                "keyword": None,
                                "page": None,
                                "status_code": None,
                                "ssl_retry_used": len(ssl_retry_urls) > ssl_retry_count_before
                            })

                    else:
                        type_b_run_stats[source_id]["fallback_checked"] = True

                time.sleep(SLEEP_TYPE_B)
                counter += 1

        # ==============================================================
        # TYPE C SITEMAP FALLBACK
        # ==============================================================
        elif url_type == "C" and RUN_TYPE_C:

            page_value = None
            base_domain = SITEMAP_FALLBACK_DOMAINS[source_id]
            full_url = base_domain

            print(f"[{source_id}] C (Sitemap fallback) | KW='{keyword}'")

            ssl_retry_count_before = len(ssl_retry_urls)
            sitemap_debug = {}

            try:
                if source_id in sitemap_url_cache:
                    all_page_urls = sitemap_url_cache[source_id]
                    sitemap_debug = sitemap_debug_cache[source_id]
                    print(f"   → Using cached sitemap URLs: {len(all_page_urls)}")
                else:
                    all_page_urls, sitemap_debug = collect_urls_for_domain(
                        base_domain,
                        source_id=source_id,
                        max_urls=10000,
                        timeout=10
                    )
                    sitemap_url_cache[source_id] = all_page_urls
                    sitemap_debug_cache[source_id] = sitemap_debug

                    print(f"   → Collected page URLs: {len(all_page_urls)}")
                    print(f"   → Sitemap source: {sitemap_debug['sitemap_source']}")
                    print(f"   → Sitemap used: {sitemap_debug['sitemap_url_used']}")
                    print(f"   → Discovery status: {sitemap_debug['discovery_status']}")

                ssl_retry_used = (
                    sitemap_debug["ssl_retry_used"] or
                    (len(ssl_retry_urls) > ssl_retry_count_before)
                )

                links = find_keyword_hits_in_urls(all_page_urls, keyword, max_hits=20)

                print(f"   → URL Hits: {len(links)} | ssl_retry_used={ssl_retry_used}")

                type_c_fallback_log.append({
                    "source_id": source_id,
                    "origin_type": "C_regular",
                    "base_domain": base_domain,
                    "keyword": keyword,
                    "status": (
                        "no_working_sitemap"
                        if sitemap_debug["discovery_status"] == "no_working_sitemap"
                        else ("hits_found" if len(links) > 0 else "no_hits")
                    ),
                    "n_hits": len(links),
                    "ssl_retry_used": ssl_retry_used,
                    "sitemap_urls_collected": len(all_page_urls),
                    "sitemap_source": sitemap_debug["sitemap_source"],
                    "sitemap_url_used": sitemap_debug["sitemap_url_used"],
                    "n_candidate_sitemaps": sitemap_debug["n_candidate_sitemaps"],
                    "robots_txt_error": sitemap_debug["robots_txt_error"],
                    "discovery_status": sitemap_debug["discovery_status"],
                    "discovery_error": sitemap_debug["discovery_error"],
                    "error": None
                })

                if len(links) == 0:
                    zero_link_urls.append({
                        "source_id": source_id,
                        "type": "C",
                        "keyword": keyword,
                        "page": page_value,
                        "search_url": full_url,
                        "ssl_retry_used": ssl_retry_used
                    })

                for link in links:
                    scraped_data.append({
                        "source_id": source_id,
                        "keyword": keyword,
                        "page": page_value,
                        "search_url": full_url,
                        "found_url": link,
                        "ssl_retry_used": ssl_retry_used
                    })

            except Exception as e:
                print(f"❌ Fehler bei Sitemap-Fallback: {full_url}")
                print(f"   → {type(e).__name__}: {e}")

                type_c_fallback_log.append({
                    "source_id": source_id,
                    "origin_type": "C_regular",
                    "base_domain": full_url,
                    "keyword": keyword,
                    "status": "fallback_failed",
                    "n_hits": None,
                    "ssl_retry_used": len(ssl_retry_urls) > ssl_retry_count_before,
                    "sitemap_urls_collected": None,
                    "sitemap_source": sitemap_debug.get("sitemap_source") if sitemap_debug else None,
                    "sitemap_url_used": sitemap_debug.get("sitemap_url_used") if sitemap_debug else None,
                    "n_candidate_sitemaps": sitemap_debug.get("n_candidate_sitemaps") if sitemap_debug else None,
                    "robots_txt_error": sitemap_debug.get("robots_txt_error") if sitemap_debug else None,
                    "discovery_status": sitemap_debug.get("discovery_status") if sitemap_debug else None,
                    "discovery_error": sitemap_debug.get("discovery_error") if sitemap_debug else None,
                    "error": f"{type(e).__name__}: {e}"
                })

                failed_urls.append({
                    "url": full_url,
                    "error": str(e),
                    "type": "C",
                    "keyword": keyword,
                    "page": page_value,
                    "status_code": None,
                    "ssl_retry_used": len(ssl_retry_urls) > ssl_retry_count_before
                })

            time.sleep(SLEEP_TYPE_C)
            counter += 1

        # ==============================================================
        # PROGRESS
        # ==============================================================
        if counter % 50 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"\n⏱️ {counter} Requests | {elapsed:.1f} min\n")

# ----------------------------------------------------------------------
# FINAL SUMMARY
# ----------------------------------------------------------------------
elapsed = (time.time() - start_time) / 60
print(f"\n⏱️ {counter} Requests | {elapsed:.1f} min\n")

print("\nAuto-promoted to sitemap fallback in this run:")

if len(auto_added_to_fallback_domains) == 0:
    print(" - none")
else:
    for source_id, original_url in auto_added_to_fallback_domains.items():
        parsed = urlparse(original_url.split("?", 1)[0].split("#", 1)[0])
        base_domain = f"{parsed.scheme}://{parsed.netloc}"
        print(f" - {source_id}: {base_domain}")

print("\nType C fallback summary:")

if len(type_c_fallback_log) == 0:
    print(" - no Type C fallback activity logged")
else:
    df_type_c_log_print = pd.DataFrame(type_c_fallback_log)

    failed_domains = sorted(
        df_type_c_log_print.loc[
            df_type_c_log_print["status"] == "fallback_failed", "source_id"
        ].dropna().unique().tolist()
    )

    no_working_sitemap_domains = sorted(
        df_type_c_log_print.loc[
            df_type_c_log_print["status"] == "no_working_sitemap", "source_id"
        ].dropna().unique().tolist()
    )

    nohit_domains = sorted(
        df_type_c_log_print.loc[
            df_type_c_log_print["status"] == "no_hits", "source_id"
        ].dropna().unique().tolist()
    )

    hit_domains = sorted(
        df_type_c_log_print.loc[
            df_type_c_log_print["status"] == "hits_found", "source_id"
        ].dropna().unique().tolist()
    )

    print(f" - domains with fallback_failed: {failed_domains if failed_domains else 'none'}")
    print(
        f" - domains with no_working_sitemap: "
        f"{no_working_sitemap_domains if no_working_sitemap_domains else 'none'}"
    )
    print(f" - domains with no_hits: {nohit_domains if nohit_domains else 'none'}")
    print(f" - domains with hits_found: {hit_domains if hit_domains else 'none'}")

[1] C (Sitemap fallback) | KW='CCS'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='CCU'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='CCUS'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='CCU/S'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='Carbon Management'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='Carbon Capture'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='Carbon Capture Utilisation and Storage'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallback) | KW='Carbon Capture Utilization and Storage'
   → Using cached sitemap URLs: 0
   → URL Hits: 0 | ssl_retry_used=False
[1] C (Sitemap fallbac

In [14]:
#11 Data frame

df = pd.DataFrame(scraped_data)
df.head(20)

,source_id,keyword,page,search_url,found_url,ssl_retry_used
0,12,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/,False
1,12,CCU,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/,False
2,12,CCUS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/,False
3,12,CCU/S,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/,False
4,12,Carbon Management,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/,False
5,12,Carbon Management,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.oekom.de/ausgabe/feldversuche-83104,False
6,12,Carbon Management,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://youtu.be/K3nT20j8Z3I,False
7,12,Carbon Management,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://youtu.be/mWaeMMapCcI,False
8,12,Carbon Capture,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/,False
9,12,Carbon Capture,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.oekom.de/ausgabe/feldversuche-83104,False


In [15]:
#12 Dataframe for failed attempts

df_failed = pd.DataFrame(failed_urls)
df_failed.head(20)



""


In [16]:
#13 Dataframe for all Domains with SSL-Retry

df_ssl_retry_domains = pd.DataFrame(
    sorted(ssl_retry_domains),
    columns=["domain"]
)

print(f"Anzahl Domains mit SSL-Retry: {len(df_ssl_retry_domains)}")
display(df_ssl_retry_domains.head(20))

Anzahl Domains mit SSL-Retry: 1


,domain
0,www.faktor3.de


In [17]:
#14 Dataframe for all URLs with SSL-Retry

df_ssl_retry_urls = pd.DataFrame(
    {"ssl_retry_url": sorted(set(ssl_retry_urls))}
)

df_ssl_retry_urls.head()

,ssl_retry_url
0,https://www.faktor3.de/case_category-sitemap.xml
1,https://www.faktor3.de/category-sitemap.xml
2,https://www.faktor3.de/customer-cases-sitemap.xml
3,https://www.faktor3.de/customer_cases_tax-site...
4,https://www.faktor3.de/job_level-sitemap.xml


In [18]:
#15 Dadtaframe for Urls with 0 result Links

df_zero = pd.DataFrame(zero_link_urls)
df_zero.head(50)

,source_id,type,keyword,page,search_url,ssl_retry_used
0,1,A,CCS,NaN,https://www.cemex.de/search?q=CCS&delta=99,False
1,1,A,CCU,NaN,https://www.cemex.de/search?q=CCU&delta=99,False
2,1,A,CCUS,NaN,https://www.cemex.de/search?q=CCUS&delta=99,False
3,1,A,CCU/S,NaN,https://www.cemex.de/search?q=CCU%2FS&delta=99,False
4,1,A,Carbon Management,NaN,https://www.cemex.de/search?q=Carbon%20Managem...,False
5,1,A,Carbon Capture,NaN,https://www.cemex.de/search?q=Carbon%20Capture...,False
6,1,A,Carbon Capture Utilisation and Storage,NaN,https://www.cemex.de/search?q=Carbon%20Capture...,False
7,1,A,Carbon Capture Utilization and Storage,NaN,https://www.cemex.de/search?q=Carbon%20Capture...,False
8,1,A,Carbon Capture and Utilisation,NaN,https://www.cemex.de/search?q=Carbon%20Capture...,False
9,1,A,Carbon Capture and Utilization,NaN,https://www.cemex.de/search?q=Carbon%20Capture...,False


In [19]:
#16 Domain Analysis for 0-Link cases

df_zero["domain"] = df_zero["search_url"].apply(lambda x: urlparse(x).netloc)
df_zero["domain"].value_counts().head(100)

domain
www.cemex.de                   57
www.vku.de                     57
suche.agora-industrie.de       57
www.lyondellbasell.com         55
www.itad.de                    47
                               ..
oge.net                         2
de.bellona.org                  2
green-planet-energy.de          2
bdi.eu                          2
www.eew-energyfromwaste.com     1
Name: count, Length: 63, dtype: int64

In [20]:
#17 Analysis of domains with zero found links across all requests

expected_cols = ["source_id", "found_url"]

if "df" not in globals():
    raise ValueError("df (scraped_data) fehlt")

# all source IDs
all_source_ids = set(urls.keys())

# source IDs with minimum one hit
source_ids_with_hits = set(df["source_id"].dropna().unique())

# difference = domains with zero hits across requests
zero_total_domains = sorted(all_source_ids - source_ids_with_hits)

# DataFrame
df_zero_total_domains = pd.DataFrame({
    "source_id": zero_total_domains
})

# add extracted domains
def extract_base_domain(url):
    if pd.isna(url):
        return None
    parsed = urlparse(url.split("?", 1)[0].split("#", 1)[0])
    return f"{parsed.scheme}://{parsed.netloc}"

df_zero_total_domains["url_template"] = df_zero_total_domains["source_id"].map(urls)
df_zero_total_domains["base_domain"] = df_zero_total_domains["url_template"].apply(extract_base_domain)

print(f"Domains mit insgesamt 0 Treffern: {len(df_zero_total_domains)}")
display(df_zero_total_domains.head(50))

Domains mit insgesamt 0 Treffern: 33


,source_id,url_template,base_domain
0,1,https://www.cemex.de/search?q={KeywordP}&delta=99,https://www.cemex.de
1,7,https://suche.agora-industrie.de/suche?tx_solr...,https://suche.agora-industrie.de
2,15,https://www.ruhr-uni-bochum.de,https://www.ruhr-uni-bochum.de
3,16,https://www.landkreistag.de/suche?q={KeywordP}...,https://www.landkreistag.de
4,17,https://www.dstgb.de/?q={KeywordP},https://www.dstgb.de
5,22,https://www.eutop.com,https://www.eutop.com
6,35,https://www.papierindustrie.de,https://www.papierindustrie.de
7,37,https://www.dr-martin-eckert.de,https://www.dr-martin-eckert.de
8,38,https://www.thyssenkrupp-uhde.com,https://www.thyssenkrupp-uhde.com
9,39,https://www.thyssenkrupp-polysius.com,https://www.thyssenkrupp-polysius.com


In [21]:
#18 Separate exclusions dataframe for filtered A/B links

expected_filtered_cols = [
    "source_id",
    "type",
    "keyword",
    "page",
    "search_url",
    "found_url",
    "filter_reason",
    "filter_stage",
    "ssl_retry_used",
]

if "filtered_out_links" not in globals():
    df_filtered_out_links = pd.DataFrame(columns=expected_filtered_cols)
else:
    df_filtered_out_links = pd.DataFrame(filtered_out_links)

    for col in expected_filtered_cols:
        if col not in df_filtered_out_links.columns:
            df_filtered_out_links[col] = pd.NA

    df_filtered_out_links = df_filtered_out_links[expected_filtered_cols].copy()

print(f"Anzahl gefilterter A/B-Links: {len(df_filtered_out_links)}")
display(df_filtered_out_links.head(30))

Anzahl gefilterter A/B-Links: 21373


,source_id,type,keyword,page,search_url,found_url,filter_reason,filter_stage,ssl_retry_used
0,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_path:/siteglobals/,safe_filter_ab,False
1,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_path:/siteglobals/,safe_filter_ab,False
2,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_path:/siteglobals/,safe_filter_ab,False
3,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_path:/siteglobals/,safe_filter_ab,False
4,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_path:/siteglobals/,safe_filter_ab,False
5,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Frontend/...,safe_file_ending:.ico,safe_filter_ab,False
6,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_path:/siteglobals/,safe_filter_ab,False
7,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_file_ending:.css,safe_filter_ab,False
8,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_file_ending:.css,safe_filter_ab,False
9,12,A,CCS,NaN,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,https://www.umweltrat.de/SiteGlobals/Forms/Suc...,safe_file_ending:.css,safe_filter_ab,False


In [22]:
#19 Dataframe for Type C fallback log

expected_type_c_cols = [
    "source_id",
    "origin_type",
    "base_domain",
    "keyword",
    "status",
    "n_hits",
    "ssl_retry_used",
    "sitemap_urls_collected",
    "sitemap_source",
    "sitemap_url_used",
    "n_candidate_sitemaps",
    "robots_txt_error",
    "discovery_status",
    "discovery_error",
    "error",
]

if "type_c_fallback_log" not in globals():
    df_type_c_log = pd.DataFrame(columns=expected_type_c_cols)
else:
    df_type_c_log = pd.DataFrame(type_c_fallback_log)

    for col in expected_type_c_cols:
        if col not in df_type_c_log.columns:
            df_type_c_log[col] = pd.NA

    df_type_c_log = df_type_c_log[expected_type_c_cols].copy()

print(f"Anzahl Type-C-Logzeilen: {len(df_type_c_log)}")
display(df_type_c_log.head(20))

Anzahl Type-C-Logzeilen: 1216


,source_id,origin_type,base_domain,keyword,status,n_hits,ssl_retry_used,sitemap_urls_collected,sitemap_source,sitemap_url_used,n_candidate_sitemaps,robots_txt_error,discovery_status,discovery_error,error
0,1,A_auto_promoted,https://www.cemex.de,CCS,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
1,1,A_auto_promoted,https://www.cemex.de,CCU,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
2,1,A_auto_promoted,https://www.cemex.de,CCUS,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
3,1,A_auto_promoted,https://www.cemex.de,CCU/S,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
4,1,A_auto_promoted,https://www.cemex.de,Carbon Management,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
5,1,A_auto_promoted,https://www.cemex.de,Carbon Capture,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
6,1,A_auto_promoted,https://www.cemex.de,Carbon Capture Utilisation and Storage,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
7,1,A_auto_promoted,https://www.cemex.de,Carbon Capture Utilization and Storage,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
8,1,A_auto_promoted,https://www.cemex.de,Carbon Capture and Utilisation,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None
9,1,A_auto_promoted,https://www.cemex.de,Carbon Capture and Utilization,no_working_sitemap,0,False,0,None,None,3,None,no_working_sitemap,ParseError: not well-formed (invalid token): l...,None


In [23]:
#20 Summary dataframe for Type C fallback outcomes by domain

expected_type_c_cols = [
    "source_id",
    "origin_type",
    "base_domain",
    "keyword",
    "status",
    "n_hits",
    "ssl_retry_used",
    "sitemap_urls_collected",
    "sitemap_source",
    "sitemap_url_used",
    "n_candidate_sitemaps",
    "robots_txt_error",
    "discovery_status",
    "discovery_error",
    "error",
]

if "df_type_c_log" not in globals():
    df_type_c_log = pd.DataFrame(columns=expected_type_c_cols)
else:
    for col in expected_type_c_cols:
        if col not in df_type_c_log.columns:
            df_type_c_log[col] = pd.NA

    df_type_c_log = df_type_c_log[expected_type_c_cols].copy()

if df_type_c_log.empty:
    df_type_c_summary = pd.DataFrame(columns=[
        "source_id",
        "base_domain",
        "origin_type",
        "final_status",
        "sitemap_source",
        "sitemap_url_used",
        "discovery_status",
        "n_keyword_checks",
        "total_hits",
        "max_sitemap_urls_collected",
        "max_candidate_sitemaps",
        "ssl_retry_used_any",
        "robots_txt_error",
        "discovery_error",
    ])
    print("Kein Type-C-Log im aktuellen Run vorhanden.")
else:
    status_rank_map = {
        "skipped_excluded": 5,
        "fallback_failed": 4,
        "no_working_sitemap": 3,
        "hits_found": 2,
        "no_hits": 1,
    }

    df_type_c_log["status_rank"] = (
        df_type_c_log["status"]
        .map(status_rank_map)
        .fillna(0)
    )

    df_type_c_summary = (
        df_type_c_log
        .groupby(
            ["source_id", "base_domain", "origin_type"],
            dropna=False,
            as_index=False
        )
        .agg(
            max_status_rank=("status_rank", "max"),
            n_keyword_checks=("keyword", "count"),
            total_hits=("n_hits", "sum"),
            max_sitemap_urls_collected=("sitemap_urls_collected", "max"),
            max_candidate_sitemaps=("n_candidate_sitemaps", "max"),
            ssl_retry_used_any=("ssl_retry_used", "max"),
        )
    )

    rank_status_map = {
        5: "skipped_excluded",
        4: "fallback_failed",
        3: "no_working_sitemap",
        2: "hits_found",
        1: "no_hits",
        0: "unknown",
    }

    df_type_c_summary["final_status"] = df_type_c_summary["max_status_rank"].map(rank_status_map)

    detail_cols = [
        "source_id",
        "base_domain",
        "origin_type",
        "status_rank",
        "status",
        "sitemap_source",
        "sitemap_url_used",
        "discovery_status",
        "robots_txt_error",
        "discovery_error",
    ]

    df_type_c_details = (
        df_type_c_log[detail_cols]
        .sort_values(
            ["source_id", "base_domain", "origin_type", "status_rank"],
            ascending=[True, True, True, False]
        )
        .drop_duplicates(subset=["source_id", "base_domain", "origin_type"])
        .drop(columns=["status_rank", "status"])
    )

    df_type_c_summary = (
        df_type_c_summary
        .merge(
            df_type_c_details,
            on=["source_id", "base_domain", "origin_type"],
            how="left"
        )
        .drop(columns=["max_status_rank"])
        .sort_values(
            ["final_status", "source_id", "origin_type"],
            ascending=[False, True, True]
        )
        .reset_index(drop=True)
    )

print(f"Anzahl Type-C-Domains in Summary: {len(df_type_c_summary)}")
display(df_type_c_summary.head(50))

Anzahl Type-C-Domains in Summary: 64


,source_id,base_domain,origin_type,n_keyword_checks,total_hits,max_sitemap_urls_collected,max_candidate_sitemaps,ssl_retry_used_any,final_status,sitemap_source,sitemap_url_used,discovery_status,robots_txt_error,discovery_error
0,1,https://www.cemex.de,A_auto_promoted,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,ParseError: not well-formed (invalid token): l...
1,1,https://www.cemex.de/search?q={KeywordP}&delta=99,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,ParseError: not well-formed (invalid token): l...
2,7,https://suche.agora-industrie.de,B_auto_promoted,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 403 Client Error: Forbidden for url...
3,7,https://suche.agora-industrie.de/suche?tx_solr...,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 403 Client Error: Forbidden for url...
4,15,https://www.ruhr-uni-bochum.de,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 404 Client Error: Not Found for url...
5,16,https://www.landkreistag.de,B_auto_promoted,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 404 Client Error: Not Found for url...
6,16,https://www.landkreistag.de/suche?q={KeywordP}...,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 404 Client Error: Not Found for url...
7,17,https://www.dstgb.de,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 404 Client Error: 404 for url: http...
8,22,https://www.eutop.com,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,None,HTTPError: 404 Client Error: Not Found for url...
9,35,https://www.papierindustrie.de,C_regular,19,0,0,3,False,no_working_sitemap,None,None,no_working_sitemap,HTTPError: 404 Client Error: Not Found for url...,HTTPError: 404 Client Error: Not Found for url...


In [24]:
#21 Error analysis / Debugging I

expected_failed_cols = ["source_id", "url", "error"]

if "df_failed" not in globals():
    df_failed = pd.DataFrame(columns=expected_failed_cols)
else:
    for col in expected_failed_cols:
        if col not in df_failed.columns:
            df_failed[col] = pd.NA

df_failed = df_failed[expected_failed_cols].copy()

print(f"Anzahl fehlgeschlagener Requests: {len(df_failed)}")

if df_failed.empty:
    print("Keine fehlgeschlagenen Requests im aktuellen Run.")
else:
    error_summary = (
        df_failed["error"]
        .fillna("unknown")
        .value_counts(dropna=False)
        .rename_axis("error")
        .reset_index(name="count")
    )
    display(error_summary)
    display(df_failed.head(20))

Anzahl fehlgeschlagener Requests: 0
Keine fehlgeschlagenen Requests im aktuellen Run.


In [25]:
#22 Error analysis / Debugging II

# domain analysis for failed URLs

expected_failed_cols = ["source_id", "url", "error"]

if "df_failed" not in globals():
    df_failed = pd.DataFrame(columns=expected_failed_cols)
else:
    for col in expected_failed_cols:
        if col not in df_failed.columns:
            df_failed[col] = pd.NA

df_failed = df_failed[expected_failed_cols].copy()

if df_failed.empty or df_failed["url"].dropna().empty:
    print("Keine fehlgeschlagenen URLs für Domain-Auswertung vorhanden.")
    failed_domain_summary = pd.DataFrame(columns=["domain", "count"])
else:
    failed_domain_summary = (
        df_failed.assign(
            domain=df_failed["url"].fillna("").apply(
                lambda x: urlparse(x).netloc if str(x).strip() else pd.NA
            )
        )
        .dropna(subset=["domain"])
        .groupby("domain", as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    print(f"Anzahl Domains mit fehlgeschlagenen Requests: {len(failed_domain_summary)}")
    display(failed_domain_summary.head(20))


Keine fehlgeschlagenen URLs für Domain-Auswertung vorhanden.


In [26]:
#23 Dataframe without URL duplicates for export

df_unique = df.drop_duplicates(subset=["found_url"]).reset_index(drop=True)

In [27]:
#24 Counter of URL prevalence / search result hit frequency for export

from collections import Counter

freq = Counter(df["found_url"])

df_freq = pd.DataFrame(freq.most_common(), columns=["url","frequency"])
df_freq.head(100) # head(zahl) sagt wie viele zeilen von dem dataframe im output gezeigt werden sollen


,url,frequency
0,https://api.whatsapp.com/send?text=https://www...,18
1,https://www.bveg.de/die-branche/erdgas-und-erd...,18
2,https://www.wwf.de/,17
3,https://www.umweltrat.de/,15
4,https://www.bveg.de/der-verband/publikationen/...,15
...,...,...
95,https://www.vbw-bayern.de/vbw/Themen-und-Servi...,8
96,https://www.vbw-bayern.de/vbw/vbw-Fokusthemen/...,8
97,https://www.vbw-bayern.de/vbw/Themen-und-Servi...,8
98,https://www.vbw-bayern.de/vbw/vbw-Fokusthemen/...,8


In [28]:
#25 Actor Categories (Input) for results export (to facilitate post-processing) (taken from Lobbyregister entries)


import pandas as pd

# Mapping: source_id -> actor_category

actor_category_map = {
    1: "Sonstiges Unternehmen",
    2: "Wirtschaftsverband oder Gewerbeverband/-verein",
    3: "Sonstiges Unternehmen",
    4: "Privatrechtliche Organisation",
    5: "Wirtschaftsverband oder Gewerbeverband/-verein",
    6: None,
    7: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    8: "Nichtregierungsorganisation (NGO)",
    9: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    10: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    11: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    12: None,
    13: "Nichtregierungsorganisation (NGO)",
    14: None,
    15: None,
    16: None,
    17: None,
    18: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    19: "Sonstiges Unternehmen",
    20: "Sonstiges Unternehmen",
    21: "Wirtschaftsverband oder Gewerbeverband/-verein",
    22: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    23: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    24: "Nichtregierungsorganisation (NGO)",
    25: "Wirtschaftsverband oder Gewerbeverband/-verein",
    26: "Wirtschaftsverband oder Gewerbeverband/-verein",
    27: "Wirtschaftsverband oder Gewerbeverband/-verein",
    28: "Privatrechtliche Organisation",
    29: "Plattform, Netzwerk, Interessengemeinschaft, Denkfabrik, Initiative, Aktionsbündnis o. ä.",
    30: "Plattform, Netzwerk, Interessengemeinschaft, Denkfabrik, Initiative, Aktionsbündnis o. ä.",
    31: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    32: "Wirtschaftsverband oder Gewerbeverband/-verein",
    33: "Wirtschaftsverband oder Gewerbeverband/-verein",
    34: "Wirtschaftsverband oder Gewerbeverband/-verein",
    35: "Wirtschaftsverband oder Gewerbeverband/-verein",
    36: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    37: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    38: "Sonstiges Unternehmen",
    39: "Sonstiges Unternehmen",
    40: "Sonstiges Unternehmen",
    41: "Sonstiges Unternehmen",
    42: "Sonstiges Unternehmen",
    43: "Sonstiges Unternehmen",
    44: "Sonstiges Unternehmen",
    45: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    46: "Sonstiges Unternehmen",
    47: "Nichtregierungsorganisation (NGO)",
    48: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    49: "Nichtregierungsorganisation (NGO)",
    50: "Nichtregierungsorganisation (NGO)",
    51: "Nichtregierungsorganisation (NGO)",
    52: "Sonstiges Unternehmen",
    53: "Plattform, Netzwerk, Interessengemeinschaft, Denkfabrik, Initiative, Aktionsbündnis o. ä.",
    54: "Sonstiges Unternehmen",
    55: "Wirtschaftsverband oder Gewerbeverband/-verein",
    56: "Wirtschaftsverband oder Gewerbeverband/-verein",
    57: "Sonstiges Unternehmen",
    58: "Sonstiges Unternehmen",
    59: "Sonstiges Unternehmen",
    60: "Wirtschaftsverband oder Gewerbeverband/-verein",
    61: "Sonstiges Unternehmen",
    62: "Sonstiges Unternehmen",
    63: "Sonstiges Unternehmen",
    64: "Sonstiges Unternehmen",
    65: "Sonstiges Unternehmen",
    66: "Sonstiges Unternehmen",
    67: "Sonstiges Unternehmen",
    68: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    69: "Sonstiges Unternehmen",
    70: "Sonstiges Unternehmen",
    71: "Sonstiges Unternehmen",
    72: "Wirtschaftsverband oder Gewerbeverband/-verein",
    73: "Sonstiges Unternehmen",
    74: "Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung",
    75: "Wirtschaftsverband oder Gewerbeverband/-verein",
    76: "Wirtschaftsverband oder Gewerbeverband/-verein",
    77: "Wirtschaftsverband oder Gewerbeverband/-verein",
    78: "Berufsverband",
    79: "Wirtschaftsverband oder Gewerbeverband/-verein",
    80: "Wirtschaftsverband oder Gewerbeverband/-verein",
    81: "Wirtschaftsverband oder Gewerbeverband/-verein",
    82: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    83: "Privatrechtliche Organisation",
    84: "Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",
    85: "Berufsverband",
    86: "Wirtschaftsverband oder Gewerbeverband/-verein",
    87: "Wirtschaftsverband oder Gewerbeverband/-verein",
    88: "Sonstiges Unternehmen",
    89: "Wirtschaftsverband oder Gewerbeverband/-verein",
}

# DataFrame for later merges
df_actor_category = pd.DataFrame(
    [
        {"source_id": k, "actor_category": v}
        for k, v in actor_category_map.items()
    ]
)

# unify data types
df_actor_category["source_id"] = pd.to_numeric(
    df_actor_category["source_id"],
    errors="coerce"
).astype("Int64")

# Check: missing categories for source_ids?
missing_ids = set(urls.keys()) - set(actor_category_map.keys())

if missing_ids:
    print("Warnung: Diese source_ids haben keinen Eintrag in actor_category_map:")
    print(sorted(missing_ids))
else:
    print("Alle source_ids aus urls haben einen Eintrag in actor_category_map.")

df_actor_category.head(90)

Alle source_ids aus urls haben einen Eintrag in actor_category_map.


,source_id,actor_category
0,1,Sonstiges Unternehmen
1,2,Wirtschaftsverband oder Gewerbeverband/-verein
2,3,Sonstiges Unternehmen
3,4,Privatrechtliche Organisation
4,5,Wirtschaftsverband oder Gewerbeverband/-verein
...,...,...
84,85,Berufsverband
85,86,Wirtschaftsverband oder Gewerbeverband/-verein
86,87,Wirtschaftsverband oder Gewerbeverband/-verein
87,88,Sonstiges Unternehmen


In [29]:
#26 URL prevalence by actor incl. category

from urllib.parse import urlparse

# --------------------------------------------------
# Safety check: required inputs present?
# --------------------------------------------------
required_vars = ["df", "df_actor_category", "urls"]

missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    raise NameError(f"Fehlende Variable(n): {missing_vars}. Bitte Reihenfolge der Zellen prüfen.")

# --------------------------------------------------
# derive base_domain from urls-Dictionary
# --------------------------------------------------
df_base_domain = pd.DataFrame(
    [
        {
            "source_id": source_id,
            "base_domain": f"{urlparse(url_template).scheme}://{urlparse(url_template).netloc}"
        }
        for source_id, url_template in urls.items()
    ]
)

df_base_domain["source_id"] = pd.to_numeric(
    df_base_domain["source_id"],
    errors="coerce"
).astype("Int64")

# --------------------------------------------------
# calculate hit frequencies per source_id + found_url
# --------------------------------------------------
df_url_freq_by_actor = (
    df.copy()
    .assign(
        source_id=lambda x: pd.to_numeric(x["source_id"], errors="coerce").astype("Int64"),
        found_url=lambda x: x["found_url"].astype(str).str.strip()
    )
    .loc[lambda x: x["found_url"].notna() & x["found_url"].ne("")]
    .groupby(["source_id", "found_url"], dropna=False, as_index=False)
    .size()
    .rename(columns={"size": "frequency"})
)

# --------------------------------------------------
# append base_domain + actor_category
# --------------------------------------------------
df_url_freq_by_actor = (
    df_url_freq_by_actor
    .merge(df_base_domain, on="source_id", how="left")
    .merge(df_actor_category, on="source_id", how="left")
    [["source_id", "base_domain", "actor_category", "found_url", "frequency"]]
    .sort_values(
        by=["source_id", "frequency", "found_url"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

# --------------------------------------------------
# Check for missing categories
# --------------------------------------------------
missing_actor_categories = (
    df_url_freq_by_actor.loc[df_url_freq_by_actor["actor_category"].isna(), "source_id"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

if missing_actor_categories:
    print("Warnung: Für folgende source_id(s) fehlt eine actor_category:")
    print(sorted(missing_actor_categories))
else:
    print("Alle source_id-Werte haben eine actor_category.")

df_url_freq_by_actor.head(100)

Warnung: Für folgende source_id(s) fehlt eine actor_category:
[6, 12, 14]


,source_id,base_domain,actor_category,found_url,frequency
0,2,https://bdi.eu,Wirtschaftsverband oder Gewerbeverband/-verein,https://bdi.eu/de/articles/energie-mobilitaet-...,13
1,2,https://bdi.eu,Wirtschaftsverband oder Gewerbeverband/-verein,https://bdi.eu/de/articles/energie-mobilitaet-...,9
2,2,https://bdi.eu,Wirtschaftsverband oder Gewerbeverband/-verein,https://bdi.eu/de/articles/energie-mobilitaet-...,9
3,2,https://bdi.eu,Wirtschaftsverband oder Gewerbeverband/-verein,https://bdi.eu/de/articles/energie-mobilitaet-...,9
4,2,https://bdi.eu,Wirtschaftsverband oder Gewerbeverband/-verein,https://bdi.eu/de/articles/energie-mobilitaet-...,8
...,...,...,...,...,...
95,4,https://green-planet-energy.de,Privatrechtliche Organisation,https://green-planet-energy.de/fileadmin/docs/...,4
96,4,https://green-planet-energy.de,Privatrechtliche Organisation,https://green-planet-energy.de/presse/artikel/...,4
97,4,https://green-planet-energy.de,Privatrechtliche Organisation,https://green-planet-energy.de/fileadmin/docs/...,3
98,4,https://green-planet-energy.de,Privatrechtliche Organisation,https://green-planet-energy.de/fileadmin/docs/...,3


In [30]:
#27 --------------------RESULTS EXPORT----------------------------------------------------------------
# Export with run-specific timestamp-named-folder (folder name includes suffix for activated URL types in the respective run)
# including Type-C-Debug-Exports and A/B-Exclusion-Export

from pathlib import Path
from datetime import datetime

# base folder
base_output_dir = Path("output_checkpoints")
base_output_dir.mkdir(parents=True, exist_ok=True)

# derive active URL types from Toggle-Switches
active_types = []
if globals().get("RUN_TYPE_A", False):
    active_types.append("A")
if globals().get("RUN_TYPE_B", False):
    active_types.append("B")
if globals().get("RUN_TYPE_C", False):
    active_types.append("C")

active_types_suffix = "".join(active_types) if active_types else "NONE"

# time stamp + Run folder name
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_folder_name = f"run_{run_timestamp}_{active_types_suffix}"

run_output_dir = base_output_dir / run_folder_name
run_output_dir.mkdir(parents=True, exist_ok=False)

# ------------------------------------------------------------------
# STANDARD EXPORTS
# ------------------------------------------------------------------

export_map = {
    "scraped_data.xlsx": globals().get("df", pd.DataFrame()),
    "failed_urls.xlsx": globals().get("df_failed", pd.DataFrame(columns=["source_id", "url", "error"])),
    "zero_link_urls.xlsx": globals().get("df_zero", pd.DataFrame()),
    "auto_added_to_fallback_domains.xlsx": globals().get("df_auto_fallback", pd.DataFrame()),
    "type_c_fallback_log.xlsx": globals().get("df_type_c_fallback", pd.DataFrame()),
    "ssl_retry_urls.xlsx": globals().get("df_ssl_retry_urls", pd.DataFrame()),
    "ssl_retry_domains.xlsx": globals().get("df_ssl_retry_domains", pd.DataFrame()),
    "filtered_out_links.xlsx": globals().get("df_filtered_out_links", pd.DataFrame()),
    "zero_total_domains.xlsx": globals().get("df_zero_total_domains", pd.DataFrame()),
    "url_frequency_by_actor.xlsx": globals().get("df_url_freq_by_actor", pd.DataFrame()),
}

# ------------------------------------------------------------------
# TYPE-C DEBUG EXPORTS
# ------------------------------------------------------------------

if "df_type_c_log" in globals():
    export_map["df_type_c_log.xlsx"] = df_type_c_log

if "df_type_c_summary" in globals():
    export_map["df_type_c_summary.xlsx"] = df_type_c_summary

# ------------------------------------------------------------------
# RUN EXPORT
# ------------------------------------------------------------------

for filename, df_export in export_map.items():
    export_path = run_output_dir / filename
    df_export.to_excel(export_path, index=False)

# ------------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------------

print("Export abgeschlossen.")
print(f"Run-Ordner: {run_output_dir.resolve()}")
print(f"Aktive Typen: {active_types_suffix}")

print("\nExportierte Dateien:")
for filename in export_map:
    print("-", filename)

Export abgeschlossen.
Run-Ordner: C:\Users\tvelthaus\OneDrive - Deutsche Energie-Agentur GmbH\MA\Text_Sample\output_checkpoints\run_2026-06-02_21-06-42_C
Aktive Typen: C

Exportierte Dateien:
- scraped_data.xlsx
- failed_urls.xlsx
- zero_link_urls.xlsx
- auto_added_to_fallback_domains.xlsx
- type_c_fallback_log.xlsx
- ssl_retry_urls.xlsx
- ssl_retry_domains.xlsx
- filtered_out_links.xlsx
- zero_total_domains.xlsx
- url_frequency_by_actor.xlsx
- df_type_c_log.xlsx
- df_type_c_summary.xlsx
